In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
from timm import create_model
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Dispositivo: {device}')


Dispositivo: cuda


In [12]:
IMG_SIZE = 70
BATCH_SIZE = 32
EPOCHS = 20
NUM_CLASSES = 7
K_FOLDS = 3

DATASET_DIR = "../data/crops/flat_7_classes"  # ajuste se necessário

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


In [13]:
dataset = datasets.ImageFolder(DATASET_DIR, transform=transform)
class_names = dataset.classes
targets = [sample[1] for sample in dataset.samples]
print(f"Classes: {class_names}")
print(f"Total de imagens: {len(dataset)}")


Classes: ['ASCH', 'ASCUS', 'HSIL', 'LSIL', 'SCC', 'nao_celula', 'sem_lesao']
Total de imagens: 22117


In [14]:
resultados = []
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
    print(f"\n=== Fold {fold+1} ===")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = create_model("convnext_tiny", pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0003)

    # Treinamento
    model.train()
    for epoch in range(EPOCHS):
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Loss médio da época {epoch+1}: {running_loss/len(train_loader):.4f}")

    # Avaliação
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())

    # Métricas
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(xticks_rotation=45, cmap="Blues")
    plt.title(f"Matriz de Confusão - Fold {fold+1}")
    os.makedirs("../results/flat_convnext", exist_ok=True)
    plt.savefig(f"../results/flat_convnext/confusao_fold{fold+1}.png")
    plt.close()

    resultados.append({
        "fold": fold+1,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    })

    print(classification_report(y_true, y_pred, target_names=class_names))



=== Fold 1 ===


Epoch 1/20: 100%|██████████| 461/461 [00:24<00:00, 18.86it/s]


Loss médio da época 1: 0.6631


Epoch 2/20: 100%|██████████| 461/461 [00:23<00:00, 19.95it/s]


Loss médio da época 2: 0.4413


...

Loss médio da época 18: 0.0370


Epoch 19/20: 100%|██████████| 461/461 [00:24<00:00, 19.17it/s]


Loss médio da época 19: 0.0325


Epoch 20/20: 100%|██████████| 461/461 [00:23<00:00, 19.23it/s]


Loss médio da época 20: 0.0376
              precision    recall  f1-score   support

        ASCH       0.84      0.27      0.41       308
       ASCUS       0.60      0.71      0.65       450
        HSIL       0.81      0.72      0.76       568
        LSIL       0.50      0.60      0.55       453
         SCC       0.81      0.84      0.83       333
  nao_celula       0.99      0.96      0.98      3000
   sem_lesao       0.86      0.92      0.89      2260

    accuracy                           0.86      7372
   macro avg       0.77      0.72      0.72      7372
weighted avg       0.87      0.86      0.86      7372



In [15]:
df_resultados = pd.DataFrame(resultados)
df_resultados.loc["Média"] = df_resultados.mean(numeric_only=True)
df_resultados.to_csv("../results/flat_convnext/metricas_finais.csv", index=False)
df_resultados


,fold,accuracy,precision,recall,f1
0,1.0,0.865048,0.766514,0.725315,0.742739
1,2.0,0.865844,0.753141,0.764304,0.757224
2,3.0,0.860689,0.773153,0.719435,0.723493
Média,2.0,0.863860,0.764269,0.736351,0.741152
